# What's 4 Dinner? — Area Builder (v2.8.0)
**ARTEZIQ**

This notebook researches restaurants in chosen areas and builds the data files the app loads automatically:

- **Real photos** — the main photo each restaurant sets on its own website (the one used for link previews).
- **Menu picks** — up to 6 healthier dishes per restaurant, taken **word-for-word** from its online menu, tagged with the app's filters (low calorie, low sodium, GLP-1 friendly, high protein, vegetarian, gluten-free, dairy-free).

**Guardrails (why this won't repeat the old notebook's mistakes)**
1. Restaurants come only from OpenStreetMap — the same real places the app shows. Nothing is invented.
2. The AI only reads menu text that was actually downloaded. Every dish it suggests is checked against that text; anything that isn't on the menu is thrown away.
3. Chains are skipped (the app handles them), and so are sites whose `robots.txt` asks bots to stay out.

**How to run**
1. `Runtime → Change runtime type → T4 GPU` (free).
2. `Runtime → Run all`. Expect roughly 1–3 hours for both areas; progress is saved as it goes, so if Colab disconnects just run all again and it picks up where it stopped (turn on `SAVE_TO_DRIVE` to survive a full reset).
3. At the end a zip downloads: `area-packs-YYYY-MM-DD.zip`. It already has the repo folders — upload its `data/areas/` files to the same place in your GitHub repo.

Rerun every few months to keep menus fresh.

In [ ]:
# 1) Settings — edit here
AREAS = [
    {
        "id": "summerville-charleston-sc",
        "name": "Summerville & Charleston, SC",
        "center": [32.90, -80.05],   # the app uses this + radius_mi to know when to load this file
        "radius_mi": 25,
        "search": [                  # where to look for restaurants
            {"name": "Summerville", "lat": 33.0185, "lon": -80.1756, "radius_mi": 12},
            {"name": "Charleston",  "lat": 32.7765, "lon": -79.9311, "radius_mi": 12},
        ],
    },
    {
        "id": "portstewart-londonderry-ni",
        "name": "Portstewart & Londonderry, Northern Ireland",
        "center": [55.09, -7.01],
        "radius_mi": 25,
        "search": [
            {"name": "Portstewart", "lat": 55.1817, "lon": -6.7195, "radius_mi": 10},
            {"name": "Londonderry", "lat": 54.9966, "lon": -7.3086, "radius_mi": 10},
        ],
    },
]

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"   # best quality on a free T4. "Qwen/Qwen2.5-3B-Instruct" is ~2x faster.
USE_AI = True                          # False = photos only (fast, no GPU needed)
MAX_PLACES_PER_AREA = 500              # safety cap
REQUEST_DELAY_S = 1.0                  # politeness delay between requests to the same website
SAVE_TO_DRIVE = False                  # True = keep progress in Google Drive (survives a full runtime reset)

APP_URL = "https://davidfliesen.github.io/Whats-For-Dinner/"
USER_AGENT = f"Whats4DinnerAreaBuilder/2.8 (+{APP_URL})"

In [ ]:
# 2) Install / import
!pip -q install rapidfuzz pypdf beautifulsoup4 lxml bitsandbytes accelerate
import os, re, io, json, time, math, datetime, zipfile, urllib.parse, urllib.robotparser, warnings
import requests
from bs4 import BeautifulSoup
from rapidfuzz import fuzz
from pypdf import PdfReader
warnings.filterwarnings("ignore")

TODAY = datetime.date.today().isoformat()
WORK = "/content/w4d"
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/w4d-area-builder"
os.makedirs(WORK, exist_ok=True)
session = requests.Session()
session.headers.update({"User-Agent": USER_AGENT, "Accept-Language": "en"})
print("Working folder:", WORK)

In [ ]:
# 3) Real restaurants from OpenStreetMap (same query the app uses)
OVERPASS = ["https://overpass-api.de/api/interpreter", "https://overpass.kumi.systems/api/interpreter"]

def overpass(lat, lon, radius_mi):
    q = (f'[out:json][timeout:60];nwr["amenity"~"^(restaurant|fast_food)$"]["name"]'
         f'(around:{round(radius_mi*1609.34)},{lat},{lon});out center tags;')
    for ep in OVERPASS:
        try:
            r = session.post(ep, data={"data": q}, timeout=90)
            if r.ok: return r.json().get("elements", [])
        except Exception as e:
            print("  Overpass retry:", e)
        time.sleep(3)
    raise RuntimeError("Overpass is unavailable right now — try again in a few minutes.")

SOCIAL_OR_PLATFORM = re.compile(r"(facebook|instagram|twitter|x\.com|tiktok|linktr\.ee|yelp\.|tripadvisor\.|google\.|goo\.gl|"
                                r"ubereats|doordash|grubhub|deliveroo|just-?eat|menulog|toasttab|square\.site|clover\.com|"
                                r"order\.online|opentable|resy\.com|sevenrooms|wix\.com/?$)", re.I)

def places_for(area):
    found = {}
    for s in area["search"]:
        print(f"Searching {s['name']} ({s['radius_mi']} mi)…")
        for el in overpass(s["lat"], s["lon"], s["radius_mi"]):
            t = el.get("tags", {})
            pid = el["type"][0] + str(el["id"])           # same id format the app uses (n123, w456, r789)
            site = t.get("website") or t.get("contact:website") or ""
            found[pid] = {"id": pid, "name": t.get("name", ""), "website": site,
                          "chain": bool(t.get("brand") or t.get("brand:wikidata")),
                          "cuisine": t.get("cuisine", "")}
        time.sleep(2)
    usable = [p for p in found.values()
              if p["website"].startswith("http") and not p["chain"] and not SOCIAL_OR_PLATFORM.search(p["website"])]
    print(f"  {len(found)} restaurants, {len(usable)} with their own website (chains skipped).")
    return usable[:MAX_PLACES_PER_AREA]

In [ ]:
# 4) Polite website reading: robots.txt, delays, size limits
_robots, _last = {}, {}

def allowed(url):
    host = urllib.parse.urlsplit(url).netloc
    if host not in _robots:
        rp = urllib.robotparser.RobotFileParser()
        try:
            r = session.get(f"{urllib.parse.urlsplit(url).scheme}://{host}/robots.txt", timeout=10)
            rp.parse(r.text.splitlines() if r.ok else [])
        except Exception:
            rp.parse([])
        _robots[host] = rp
    return _robots[host].can_fetch(USER_AGENT, url)

def get(url, max_bytes=4_000_000):
    if not allowed(url): return None
    host = urllib.parse.urlsplit(url).netloc
    wait = REQUEST_DELAY_S - (time.time() - _last.get(host, 0))
    if wait > 0: time.sleep(wait)
    _last[host] = time.time()
    try:
        r = session.get(url, timeout=20, stream=True, allow_redirects=True)
        if not r.ok: return None
        data = r.raw.read(max_bytes, decode_content=True)
        return {"url": r.url, "type": r.headers.get("Content-Type", ""), "body": data}
    except Exception:
        return None

def looks_like_photo(url):
    """Keep real photos; skip tiny images and logos/icons."""
    if not url.startswith("https://") or re.search(r"(logo|icon|favicon|sprite|placeholder|default)", url, re.I):
        return False
    try:
        r = session.get(url, timeout=15, stream=True)
        ok = r.ok and r.headers.get("Content-Type", "").startswith("image/")
        size = int(r.headers.get("Content-Length") or 0) or len(r.raw.read(60_000))
        r.close()
        return ok and size >= 25_000
    except Exception:
        return False

def site_photo(soup, base):
    for sel in [("meta", {"property": "og:image:secure_url"}), ("meta", {"property": "og:image"}),
                ("meta", {"name": "twitter:image"}), ("meta", {"name": "twitter:image:src"})]:
        tag = soup.find(*sel)
        if tag and tag.get("content"):
            url = urllib.parse.urljoin(base, tag["content"].strip())
            if looks_like_photo(url): return url
    return ""

MENU_WORDS = re.compile(r"\b(menu|menus|food|dinner|lunch|eat|carte|dishes)\b", re.I)
PRICE = re.compile(r"(\$|£|€)\s?\d|\b\d{1,3}[.,]\d{2}\b")

def page_text(res):
    if not res: return ""
    if "pdf" in res["type"] or res["url"].lower().endswith(".pdf"):
        try:
            pdf = PdfReader(io.BytesIO(res["body"]))
            return "\n".join((pg.extract_text() or "") for pg in pdf.pages[:12])
        except Exception:
            return ""
    soup = BeautifulSoup(res["body"], "lxml")
    for bad in soup(["script", "style", "noscript", "svg", "nav", "footer", "form"]): bad.decompose()
    return soup.get_text("\n")

def clean_lines(text, limit=12000):
    seen, out = set(), []
    for line in (l.strip() for l in text.splitlines()):
        line = re.sub(r"\s+", " ", line)
        if 2 < len(line) < 220 and line.lower() not in seen:
            seen.add(line.lower()); out.append(line)
    return "\n".join(out)[:limit]

def read_site(p):
    home = get(p["website"])
    if not home or "html" not in home["type"]: return {"photo": "", "menu": "", "menuUrl": ""}
    soup = BeautifulSoup(home["body"], "lxml")
    photo = site_photo(soup, home["url"])
    host = urllib.parse.urlsplit(home["url"]).netloc.replace("www.", "")
    links = []
    for a in soup.find_all("a", href=True):
        href = urllib.parse.urljoin(home["url"], a["href"])
        label = (a.get_text(" ") or "") + " " + href
        if host in href and MENU_WORDS.search(label) and not href.startswith("mailto:"):
            links.append(href.split("#")[0])
    links = list(dict.fromkeys(links))[:3]
    texts, menu_url = [page_text(home)], ""
    for href in links:
        t = page_text(get(href))
        if len(PRICE.findall(t)) >= 4 and not menu_url: menu_url = href
        texts.append(t)
    menu = clean_lines("\n".join(texts))
    if len(PRICE.findall(menu)) < 4: menu = ""          # not a real menu (no prices) — don't ask the AI
    return {"photo": photo, "menu": menu, "menuUrl": menu_url or (links[0] if links else "")}

In [ ]:
# 5) The AI reader (runs locally on the free T4 — no API keys)
FILTERS = ["lowCal", "lowSodium", "glp1", "highProtein", "vegetarian", "glutenFree", "dairyFree"]
model = tok = None
if USE_AI:
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    tok = AutoTokenizer.from_pretrained(MODEL_ID)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, device_map="auto",
        quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16))
    model.eval()
    print("Loaded", MODEL_ID)

PROMPT = """Restaurant: {name}
Menu text (copied from their website, may be messy):
<<<
{menu}
>>>
Choose up to 6 of the HEALTHIER dishes on this menu. Prefer grilled, baked, broiled, steamed or raw dishes; lean protein (fish, chicken, lean meat, eggs, beans, tofu); vegetables and salads; modest portions. Never choose anything fried, breaded, battered, creamy, cheesy, smothered, or loaded with cured meat.
For each dish, list which tags it likely fits:
- lowCal: a lighter dish (roughly under 450 calories)
- lowSodium: no cured meats, soy/teriyaki, heavy sauces, soups, pickles or lots of cheese
- glp1: lean protein, low fat, not fried, modest portion
- highProtein: a substantial portion of meat, fish, eggs, tofu or beans
- vegetarian: no meat or fish
- glutenFree: no bread, pasta, breading, flour-thickened sauce or soy sauce
- dairyFree: no cheese, cream, butter sauce or yogurt
Rules: copy each dish name EXACTLY as it is written in the menu text. Only use dishes that are in the text. If this is not a food menu, return {{"dishes": []}}.
Reply with JSON only, like: {{"dishes": [{{"dish": "Grilled Salmon", "note": "grilled, served with vegetables", "fits": ["lowCal","glp1","highProtein","glutenFree"]}}]}}"""

def ask_ai(name, menu):
    msgs = [{"role": "system", "content": "You read restaurant menus carefully and never invent dishes."},
            {"role": "user", "content": PROMPT.format(name=name, menu=menu)}]
    ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=600, do_sample=False, pad_token_id=tok.eos_token_id)
    text = tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True)
    m = re.search(r"\{.*\}", text, re.S)
    try:
        return json.loads(m.group(0)).get("dishes", []) if m else []
    except Exception:
        return []

norm = lambda s: re.sub(r"[^a-z0-9 ]+", " ", str(s).lower()).strip()

def verified_picks(raw, menu):
    """Keep only dishes that really appear in the downloaded menu text."""
    menu_n, picks, seen = norm(menu), [], set()
    for d in raw if isinstance(raw, list) else []:
        dish = str(d.get("dish", "")).strip() if isinstance(d, dict) else ""
        if len(dish) < 3 or norm(dish) in seen: continue
        if norm(dish) not in menu_n and fuzz.partial_ratio(norm(dish), menu_n) < 92: continue
        seen.add(norm(dish))
        fits = [f for f in d.get("fits", []) if f in FILTERS] if isinstance(d.get("fits"), list) else []
        picks.append({"dish": dish[:120], "note": str(d.get("note", ""))[:160], "fits": fits})
    return picks[:6]

In [ ]:
# 6) Build each area (resumes automatically if interrupted)
def build(area):
    cache_path = f"{WORK}/{area['id']}.progress.json"
    done = json.load(open(cache_path)) if os.path.exists(cache_path) else {}
    todo = [p for p in places_for(area) if p["id"] not in done]
    print(f"{len(done)} already done, {len(todo)} to go.")
    for i, p in enumerate(todo, 1):
        info = read_site(p)
        picks = verified_picks(ask_ai(p["name"], info["menu"]), info["menu"]) if (USE_AI and info["menu"]) else []
        done[p["id"]] = {"name": p["name"], "website": p["website"], "photo": info["photo"],
                         "photoSource": p["website"] if info["photo"] else "", "menuUrl": info["menuUrl"],
                         "picks": picks, "checked": TODAY}
        print(f"  [{i}/{len(todo)}] {p['name'][:40]:40} photo:{'yes' if info['photo'] else '—':3}  picks:{len(picks)}")
        if i % 5 == 0: json.dump(done, open(cache_path, "w"))
    json.dump(done, open(cache_path, "w"))
    keep = {k: v for k, v in done.items() if v["photo"] or v["picks"]}
    meta = {k: area[k] for k in ("id", "name", "center", "radius_mi")}
    meta["file"] = f"data/areas/{area['id']}.json"
    meta["generated"] = TODAY
    return {"version": 1, "area": meta, "generated": TODAY, "places": keep}

packs_out = []
for area in AREAS:
    print("\n=== " + area["name"] + " ===")
    pack = build(area)
    packs_out.append(pack)
    ph = sum(1 for v in pack["places"].values() if v["photo"])
    pk = sum(1 for v in pack["places"].values() if v["picks"])
    print(f"Done: {len(pack['places'])} restaurants enriched — {ph} with a real photo, {pk} with menu picks.")

In [ ]:
# 7) Save with the repo folder structure and download
out_dir = "/content/out/data/areas"
os.makedirs(out_dir, exist_ok=True)
for pack in packs_out:
    json.dump(pack, open(f"{out_dir}/{pack['area']['id']}.json", "w"), ensure_ascii=False, separators=(",", ":"))
json.dump({"version": 1, "generated": TODAY, "areas": [p["area"] for p in packs_out]},
          open(f"{out_dir}/index.json", "w"), ensure_ascii=False, indent=1)

zip_path = f"/content/area-packs-{TODAY}.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for name in sorted(os.listdir(out_dir)):
        z.write(f"{out_dir}/{name}", f"data/areas/{name}")
print("Files:", sorted(os.listdir(out_dir)))
try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    print("Download from:", zip_path)

### Checking the results before you publish
- Open the app → **How we check these** → **Import JSON** and pick one of the area files. It loads on your device only, so you can look it over.
- If a restaurant asks to be removed, delete its entry from the area file (search for its name) and upload again.